# HydraNet Quick Start

This notebook demonstrates:
1. Loading pre-trained model weights from HuggingFace
2. Splitting the model into ENCODER, BOTTLENECK, and DECODER components
3. Exporting ENCODER, BOTTLENECK, and DECODER to ONNX
4. Exporting the full model with `hydranet.export.ModelExporter`, validating the ONNX artifact, and preparing OpenVINO 2020.3-compatible IR conversion

**Repository:** `sirbastiano/hydranet-phisat2`  
**Available:** 139 pre-trained model configurations

## 1. Load Model with Pre-trained Weights

Load the student model with automatic weight download from HuggingFace.

In [1]:
import hydranet

# Configuration
task = 'burned_area'
n_shots = 5000
training = 'finetuning'

# Load model with automatic weight download and loading
model = hydranet.load_student(
    preset='checkpoint',      # Matches HF checkpoint architecture
    task=task,
    n_shots=n_shots,
    training=training,
    auto_load_weights=True,
    checkpoint_selection='best',  # Select by HF artifacts metrics instead of newest-only
    weights_dir='../weights',
    strict=False             # Allows task-specific head mismatches
)

print(f"✓ Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

Found latest checkpoint from 20251216
  Downloading: UNet_Myriad2_Downstream_frozen_best.pt
  Saved to: ../weights/linear_probing/hydranet/burned_area_nshot5000_frozen/burned_area/20251216_UNet_Myriad2_Downstream_frozen_5000/UNet_Myriad2_Downstream_frozen_best.pt
Loading weights from: ../weights/linear_probing/hydranet/burned_area_nshot5000_frozen/burned_area/20251216_UNet_Myriad2_Downstream_frozen_5000/UNet_Myriad2_Downstream_frozen_best.pt
  Adjusted n_classes to checkpoint head: 4
✓ Model loaded: 351,172 parameters


### Checkpoint Compatibility Notes

When loading weights from Hugging Face checkpoints (`auto_load_weights=True`), HydraNet applies compatibility fixes:

- `checkpoint_selection='best'` picks the checkpoint with the strongest available metrics from each run `artifacts.json` (with fallback to latest).
- Legacy checkpoint head keys `classifier.weight` / `classifier.bias` are mapped to `final_conv.weight` / `final_conv.bias`.
- If `n_classes` is not explicitly set and checkpoint head classes differ from the local model head, `final_conv` is automatically resized to match the checkpoint output classes.
- Missing `*.convnext_block.gamma` tensors in older checkpoints are filled from model defaults.

This ensures component exports (encoder, bottleneck, decoder) use the same loaded weights as the checkpoint-compatible model.


## 2. Split Model into Components

Decompose the model into ENCODER, BOTTLENECK, and DECODER components.

In [2]:
# Split the model into components
components = hydranet.split_model(model)

# Display summary
print(components)

ModelComponents(
  ENCODER: 2 layer groups, 52,304 parameters (14.9%)
  BOTTLENECK: 1 layer groups, 146,816 parameters (41.8%)
  DECODER: 3 layer groups, 152,052 parameters (43.3%)
  TOTAL: 351,172 parameters
)


In [3]:
# Access individual components
encoder = components.encoder
bottleneck = components.bottleneck
decoder = components.decoder

print("ENCODER:", list(encoder.keys()))
print("BOTTLENECK:", list(bottleneck.keys()))
print("DECODER:", list(decoder.keys()))

ENCODER: ['encoders', 'pools']
BOTTLENECK: ['bottleneck']
DECODER: ['upsamplers', 'decoders', 'final_conv']


In [4]:
encoder

ModuleDict(
  (encoders): ModuleList(
    (0): ConvBlock(
      (channel_proj): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
      (convnext_block): ConvNeXtBlock(
        (dwconv): Conv2d(16, 16, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=16)
        (norm): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (pwconv1): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1))
        (act): GELU(approximate='none')
        (pwconv2): Conv2d(64, 16, kernel_size=(1, 1), stride=(1, 1))
        (drop_path): Identity()
      )
    )
    (1): ConvBlock(
      (channel_proj): Conv2d(16, 32, kernel_size=(1, 1), stride=(1, 1))
      (convnext_block): ConvNeXtBlock(
        (dwconv): Conv2d(32, 32, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=32)
        (norm): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (pwconv1): Conv2d(32, 128, kernel_size=(1, 1), stride=(1, 1))
        (act): GELU(appro

## 3. Export Components to ONNX

Export each component (`encoder`, `bottleneck`, `decoder`) as a standalone ONNX model.

In [5]:
from pathlib import Path
import torch
import torch.nn as nn

output_dir = Path('../onnx/components')
output_dir.mkdir(parents=True, exist_ok=True)

model.eval()

# components.* are ModuleDict objects, so we expose explicit forward() wrappers
class EncoderExportWrapper(nn.Module):
    def __init__(self, encoders, pools):
        super().__init__()
        self.encoders = encoders
        self.pools = pools

    def forward(self, x):
        skips = []
        current = x
        for i, enc in enumerate(self.encoders):
            current = enc(current)
            skips.append(current)
            if i < len(self.encoders) - 1:
                current = self.pools[i](current)
        bottleneck_in = self.pools[-1](current)
        return (bottleneck_in, *skips)


class DecoderExportWrapper(nn.Module):
    def __init__(self, upsamplers, decoders, final_conv):
        super().__init__()
        self.upsamplers = upsamplers
        self.decoders = decoders
        self.final_conv = final_conv

    def forward(self, bottleneck_out, *skips):
        current = bottleneck_out
        depth = len(self.decoders)
        for i in range(depth):
            current = self.upsamplers[i](current)
            skip = skips[depth - 1 - i]
            # Keep opset 10 export free of Resize/Upsample: this model at 224x224 already matches skip sizes.
            current = torch.cat([current, skip], dim=1)
            current = self.decoders[i](current)
        return self.final_conv(current)


encoder_wrapper = EncoderExportWrapper(encoder['encoders'], encoder['pools']).eval()
bottleneck_wrapper = bottleneck['bottleneck'].eval()
decoder_wrapper = DecoderExportWrapper(
    decoder['upsamplers'],
    decoder['decoders'],
    decoder['final_conv'],
).eval()

dummy_input = torch.randn(1, 8, 224, 224)

with torch.no_grad():
    encoder_outputs = encoder_wrapper(dummy_input)
    bottleneck_input = encoder_outputs[0]
    skips = encoder_outputs[1:]
    bottleneck_output = bottleneck_wrapper(bottleneck_input)

component_paths = {
    'encoder': output_dir / 'encoder.onnx',
    'bottleneck': output_dir / 'bottleneck.onnx',
    'decoder': output_dir / 'decoder.onnx',
}

# 1) Encoder export: output = bottleneck_input + all skip tensors
torch.onnx.export(
    encoder_wrapper,
    dummy_input,
    str(component_paths['encoder']),
    export_params=True,
    opset_version=10,
    input_names=['input'],
    output_names=['bottleneck_input'] + [f'skip_{i}' for i in range(len(skips))],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'bottleneck_input': {0: 'batch_size'},
        **{f'skip_{i}': {0: 'batch_size'} for i in range(len(skips))},
    },
)

# 2) Bottleneck export: input = bottleneck_input, output = bottleneck_output
torch.onnx.export(
    bottleneck_wrapper,
    bottleneck_input,
    str(component_paths['bottleneck']),
    export_params=True,
    opset_version=10,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
)

# 3) Decoder export: inputs = bottleneck_output + skip tensors
decoder_input_names = ['bottleneck_output'] + [f'skip_{i}' for i in range(len(skips))]
decoder_dynamic_axes = {name: {0: 'batch_size'} for name in decoder_input_names}
decoder_dynamic_axes['output'] = {0: 'batch_size'}

torch.onnx.export(
    decoder_wrapper,
    (bottleneck_output, *skips),
    str(component_paths['decoder']),
    export_params=True,
    opset_version=10,
    input_names=decoder_input_names,
    output_names=['output'],
    dynamic_axes=decoder_dynamic_axes,
)

for name, path in component_paths.items():
    print(f'{name}: {path.resolve()}')


encoder: /Users/roberto.delprete/Library/CloudStorage/OneDrive-ESA/Desktop/Repos/phisat2/Hydra/onnx/components/encoder.onnx
bottleneck: /Users/roberto.delprete/Library/CloudStorage/OneDrive-ESA/Desktop/Repos/phisat2/Hydra/onnx/components/bottleneck.onnx
decoder: /Users/roberto.delprete/Library/CloudStorage/OneDrive-ESA/Desktop/Repos/phisat2/Hydra/onnx/components/decoder.onnx


## 4. Export Full Model with `ModelExporter`

Use `src/hydranet/export.py` to export the complete model to ONNX with an explicit OpenVINO 2020.3 compatibility contract.

- `ModelExporter` defaults to ONNX opset 11, which is the highest opset supported by the OpenVINO 2020.3 Docker flow used in this repository.
- The exporter runs an ONNX checker pass after export so an invalid graph is rejected before OpenVINO conversion starts.
- OpenVINO conversion expects Docker plus the `openvino/ubuntu18_dev:2020.3` image, and `host_mount_root` must point at the mounted repository root seen by the container.


In [6]:
from pathlib import Path
import logging
from types import SimpleNamespace
import shutil

from hydranet.export import ModelExporter


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for candidate in candidates:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not resolve repository root from current working directory.')


repo_root = find_repo_root()
input_channels = int(dummy_input.shape[1])
input_size = int(dummy_input.shape[-1])

export_config = SimpleNamespace(n_channels=input_channels, input_size=input_size)
experiment_dir = repo_root / 'outputs' / 'notebook_export'

logger = logging.getLogger('hydranet.notebook.export')
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter('%(levelname)s | %(message)s'))
    logger.addHandler(handler)

exporter = ModelExporter(
    config=export_config,
    experiment_dir=experiment_dir,
    logger=logger,
    host_mount_root=repo_root,
    opset_version=11,
)

onnx_model_path, onnx_conversion_time = exporter.convert_to_onnx(model)
if onnx_model_path is None:
    raise RuntimeError('ONNX export failed. Check logs above.')

print(f'ONNX model path: {onnx_model_path}')
print(f'ONNX conversion time: {onnx_conversion_time:.2f} s')
print(f"Sample input path: {(exporter.onnx_dir / 'sample_input.npy').resolve()}")

# Keep this at opset 11 for OpenVINO 2020.3 compatibility.
run_openvino = False
if run_openvino:
    if shutil.which('docker') is None:
        print('OpenVINO conversion skipped: docker not available in PATH.')
        openvino_model_path, openvino_conversion_time = None, 0.0
    else:
        openvino_model_path, openvino_conversion_time = exporter.convert_to_openvino()
        print(f'OpenVINO model path: {openvino_model_path}')
        print(f'OpenVINO conversion time: {openvino_conversion_time:.2f} s')
else:
    openvino_model_path, openvino_conversion_time = None, 0.0
    print('OpenVINO conversion skipped (set run_openvino=True to enable).')


INFO | === ONNX Conversion Phase ===
INFO | === Model Export Diagnostics ===
INFO | PyTorch version: 2.4.0
INFO | Model parameters: 351,172
INFO | Model memory: 1.34 MB
INFO | Starting ONNX export with opset version 11
INFO | Model input shape: (1, 8, 224, 224)
INFO | Model input dtype: torch.float32
INFO | Forward pass successful. Output shape: (1, 4, 224, 224)
INFO | Output dtype: torch.float32
INFO | Output value range: [-1.4150, 1.3727]
INFO | Model successfully exported to /Users/roberto.delprete/Library/CloudStorage/OneDrive-ESA/Desktop/Repos/phisat2/Hydra/outputs/notebook_export/onnx/model.onnx
INFO | ONNX checker passed. Exported opset: 11
INFO | ONNX model size: 1.36 MB
INFO | Saved dummy input with shape (1, 8, 224, 224) to /Users/roberto.delprete/Library/CloudStorage/OneDrive-ESA/Desktop/Repos/phisat2/Hydra/outputs/notebook_export/onnx/sample_input.npy
INFO | Input data type: float32
INFO | Input value range: [-4.6233, 4.9127]


ONNX model path: /Users/roberto.delprete/Library/CloudStorage/OneDrive-ESA/Desktop/Repos/phisat2/Hydra/outputs/notebook_export/onnx/model.onnx
ONNX conversion time: 0.20 s
Sample input path: /Users/roberto.delprete/Library/CloudStorage/OneDrive-ESA/Desktop/Repos/phisat2/Hydra/outputs/notebook_export/onnx/sample_input.npy
OpenVINO conversion skipped (set run_openvino=True to enable).


In [8]:
import hashlib
import numpy as np
import onnx
import torch

onnx_path = Path(onnx_model_path)
sample_input_path = exporter.onnx_dir / 'sample_input.npy'

assert onnx_path.exists(), f'Missing ONNX model: {onnx_path}'
assert sample_input_path.exists(), f'Missing sample input: {sample_input_path}'

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
onnx_opset = max(entry.version for entry in onnx_model.opset_import if entry.domain in ('', 'ai.onnx'))

sample_input = np.load(sample_input_path)
expected_shape = (1, input_channels, input_size, input_size)
assert tuple(sample_input.shape) == expected_shape, (sample_input.shape, expected_shape)

with torch.no_grad():
    torch_output = model(torch.from_numpy(sample_input))

onnx_sha256 = hashlib.sha256(onnx_path.read_bytes()).hexdigest()
print('ONNX checker: PASS')
print(f'ONNX opset: {onnx_opset}')
print(f'Sample input shape: {sample_input.shape}')
print(f'PyTorch output shape on saved sample input: {tuple(torch_output.shape)}')
print(f'ONNX SHA256: {onnx_sha256}')


ONNX checker: PASS
ONNX opset: 11
Sample input shape: (1, 8, 224, 224)
PyTorch output shape on saved sample input: (1, 4, 224, 224)
ONNX SHA256: ee56140f28b7e38bed03769815f3906931f241af22ac97c4120e22490cfcb7ab
